# Desafio técnico Bradesco DE 

**Autor:** Felipe Piva

**Dataset:** Activity recognition exp.zip (4 CSVs: Phones/Watch x accelerometer/gyroscope)

## Estrutura do notebook (ETL)

0. Setup          | imports, paths, config        | 

1. RAW - Ingestão |  Leitura com schema explicito |

## Seção 0 - Setup 

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.window import Window

# Caminho para o dataset
RAW_PATH = '/Volumes/bradesco_desafio/desafio/raw/'

## Seção 1 - ingestão 


In [0]:
# Schema explícito (sem inferSchema)
schema = StructType([
    StructField("Index",         LongType(),   True),
    StructField("Arrival_Time",  LongType(),   True),
    StructField("Creation_Time", LongType(),   True),
    StructField("x",             DoubleType(), True),
    StructField("y",             DoubleType(), True),
    StructField("z",             DoubleType(), True),
    StructField("User",          StringType(), True),
    StructField("Model",         StringType(), True),
    StructField("Device",        StringType(), True),
    StructField("gt",            StringType(), True),
])

# mapa de arquivos (sensor + tipo de device)
files = {
    "Phones_accelerometer.csv" : ("phone", "accelerometer"),
    "Phones_gyroscope.csv"     : ("phone", "gyroscope"),
    "Watch_accelerometer.csv"  : ("watch", "accelerometer"),
    "Watch_gyroscope.csv"      : ("watch", "gyroscope")
}

def read_csv(
    filename: str,
    device_type: str,
    sensor : str
):
    return (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(f"{RAW_PATH}{filename}")
        .withColumn("device_type", F.lit(device_type))
        .withColumn("sensor",      F.lit(sensor))
        .withColumn("source_file", F.lit(filename))
    )

df_raw = None

for filename, (device_type, sensor) in files.items():
    df = read_csv(filename, device_type, sensor)
    df_raw = df if df_raw is None else df_raw.unionByName(df)

print(f"Total de registros lidos: {df_raw.count()}")

## Seção 2 - Análise inicial / Problemas encontrados

In [0]:
display(df_raw)

In [0]:
# Problema 1 - Coluna target 'gt' com nulos e a string literal "null"
(
    df_raw
    .groupBy("gt")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

print(f"Total de registros reais null: {df_raw.filter(F.col("gt").isNull()).count()}")
print(f"Total de registro com string null: {df_raw.filter(F.col("gt") == "null").count()}")

In [0]:
# Problema 2 - Inconsitência na Creation Date enunciado afirma que os dados são gravados em nanosegundos mas na pratica alguns devices gravam em milisegundos
# Comparar com o Arrival Time revela isso
(
    df_raw
    .groupBy("Device")
    .agg(
        F.min("Creation_Time").alias("min_creation"),
        F.max("Creation_Time").alias("max_creation"),
        F.min("Arrival_Time").alias("min_arrival"),
        F.max("Arrival_Time").alias("max_arrival"),
    )
    .show(50, truncate=False)
)

In [0]:
# Problema 3 - Dados com mesmo timestamp e mesmo user e device - indicando dados dupllicados 
dups = (
    df_raw
    .groupBy("User", "Device", "Creation_Time", "source_file")
    .count()
    .filter(F.col("count") > 1)
    .select("User", "Device", "Creation_Time", "source_file")
)

df_dups = (
    df_raw
    .join(dups, on=["User", "Device", "Creation_Time", "source_file"], how="inner")
)

display(df_dups)

In [0]:
# Problema 4 - Arrival_time maior que Creation_Time, indicando possível problema na sincronização de devices 
(
    df_raw
    .filter(F.col("Arrival_Time") > F.col("Creation_Time"))
    .groupBy("Device")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

### Resumo dos problemas e como tratar: 

- **Problema 1:** Coluna target (`gt`) com valores faltantes — a string literal "null" 
  convive com nulos reais (registros de transição entre atividades). 
  Solução: normalizar "null" → NULL; manter ou filtrar conforme o uso final.

- **Problema 2:** `Creation_Time` com inconsistência na ordem de grandeza entre devices 
  (ms vs ns). Solução: normalizar tudo para ms; usar `Arrival_Time` como referência confiável.

- **Problema 3:** Duplicatas na chave de identificação dos registros. 
  Solução: de-duplicar (remover) os registros repetidos, garantindo consistência na base final.

- **Problema 4:** Registros com `Arrival_Time` < `Creation_Time` (dado recebido antes de criado), 
  indicando problema de sincronização de relógio entre devices. 
  Solução: flagar/quarentenar esses registros; investigar o offset por device.